# AIMD-L Coordinate Transformer

This notebook demonstrates how to convert coordinates between the instrument-specific frames used by MAXIMA, HELIX, and SPHINX and the canonical sample coordinate system used across the AIMD-L facility.

The core problem: each instrument has its own stage with its own origin and axis orientation. When you measure a feature at position `(-14, -20)` in MAXIMA's frame, you need to know where that same feature is in SPHINX's frame to run a follow-up measurement. The `CoordinateTransformer` handles this by fitting affine transforms from calibration point pairs.

## Setup

Install the package if not already installed. The `[notebook]` extra includes Jupyter and ipykernel.

In [ ]:
%pip install -q coordinate-transformer[notebook]

In [ ]:
from pathlib import Path

import numpy as np

from coordinate_transformer import CoordinateTransformer

## Load the calibration configuration

The YAML file defines the canonical sample coordinate system and the calibration point pairs for each instrument. Each instrument needs at least three non-collinear calibration pairs that map `(x_instrument, y_instrument)` to `(x_sample, y_sample)`.

In [ ]:
config_path = Path("instrument_coordinate_transforms.yaml")
transformer = CoordinateTransformer.from_yaml(config_path)

## The canonical sample coordinate system

All instruments transform into a shared coordinate system defined on the sample. The convention is:

- **Origin**: bottom-left corner of the sample
- **X positive**: rightward (so X = 40 is the right edge)
- **Y positive**: upward (so Y = 40 is the top edge)
- **Units**: millimeters
- **Sample dimensions**: nominall 40 mm × 40 mm

A point at `(0, 0)` is the bottom-left corner; `(40, 40)` is the top-right corner.

In [ ]:
transformer.canonical_system

In [ ]:
print("Available instruments:", transformer.instruments())

## Validate the calibration fit

`validate()` returns the maximum residual error (in mm) for each instrument's calibration points. With exactly three calibration points, the affine fit is exact and the error should be effectively zero. Nonzero errors would indicate either noisy calibration data or an overdetermined fit with more than three points.

In [ ]:
transformer.validate()

## Transform a single point

Convert an instrument coordinate to the sample frame. This example uses the first MAXIMA calibration point `(-14, -20)`, which should map to sample origin `(0, 0)`.

In [ ]:
x_sample, y_sample = transformer.transform("MAXIMA", -14.0, -20.0)
print(f"MAXIMA (-14.0, -20.0) → sample ({x_sample:.4f}, {y_sample:.4f})")

## Inverse transform

Go the other direction: given a position in sample coordinates, find where to drive the instrument stage. Here we ask where sample position `(20, 20)` (center of the sample) falls in each instrument's frame.

In [ ]:
for name in transformer.instruments():
    x_inst, y_inst = transformer.inverse_transform(name, 20.0, 20.0)
    print(f"  sample (20, 20) → {name:7s} ({x_inst:9.4f}, {y_inst:9.4f})")

## Round-trip verification

A basic sanity check: transform a point forward into sample coordinates, then inverse-transform it back. The result should match the original within floating-point precision.

In [ ]:
x_orig, y_orig = 150.0, 120.0

x_sample, y_sample = transformer.transform("SPHINX", x_orig, y_orig)
x_back, y_back = transformer.inverse_transform("SPHINX", x_sample, y_sample)

print(f"Original:      ({x_orig}, {y_orig})")
print(f"Sample frame:  ({x_sample:.6f}, {y_sample:.6f})")
print(f"Back to SPHINX: ({x_back:.6f}, {y_back:.6f})")
print(f"Round-trip error: {abs(x_back - x_orig):.2e}, {abs(y_back - y_orig):.2e}")

## Cross-instrument translation

The most common use case: you have a coordinate from one instrument and need to find the equivalent position on another instrument's stage. This goes through the sample frame as an intermediate step.

Example: a feature was measured at `(-14, -20)` in MAXIMA. Where should the SPHINX stage go to reach the same spot?

In [ ]:
# Step 1: MAXIMA instrument coordinates → sample coordinates
x_sample, y_sample = transformer.transform("MAXIMA", -14.0, -20.0)
print(f"MAXIMA (-14, -20) → sample ({x_sample:.4f}, {y_sample:.4f})")

# Step 2: sample coordinates → SPHINX instrument coordinates
x_sphinx, y_sphinx = transformer.inverse_transform("SPHINX", x_sample, y_sample)
print(f"sample ({x_sample:.4f}, {y_sample:.4f}) → SPHINX ({x_sphinx:.4f}, {y_sphinx:.4f})")

## Batch transform

Transform multiple points at once using a numpy array. This is more efficient than looping over single-point transforms. The input is an `(N, 2)` array; the output has the same shape.

In [ ]:
sphinx_points = np.array([
    [125.319, 148.213],
    [124.924, 107.753],
    [165.189, 107.350],
    [145.0, 128.0],
])

sample_points = transformer.transform_points("SPHINX", sphinx_points)

print("SPHINX → Sample")
for inst, samp in zip(sphinx_points, sample_points, strict=False):
    print(f"  ({inst[0]:9.3f}, {inst[1]:9.3f}) → ({samp[0]:7.3f}, {samp[1]:7.3f})")

## Inspect the affine matrix

Each instrument's transform is a 3×3 affine matrix (the last row is always `[0, 0, 1]`). The 2×2 upper-left block encodes rotation, scaling, reflection, and shear. The third column encodes translation.

You can access the `InstrumentTransform` object directly for lower-level inspection.

In [ ]:
for name in transformer.instruments():
    t = transformer.get_transform(name)
    det = np.linalg.det(t.matrix[:2, :2])
    print(f"{name} (units: {t.units}, det: {det:+.4f})")
    print(t.matrix)
    print()

A negative determinant means the transform includes a reflection (the instrument's axis handedness is flipped relative to the sample frame). This is normal for instruments where the stage convention mirrors the sample convention.

## Full summary

The `summary()` method returns a dictionary with the canonical system definition, each instrument's fitted matrix, and its calibration error — useful for logging or serialization.

In [ ]:
import json

print(json.dumps(transformer.summary(), indent=2))